# Week 6 Period-Specific Isolation Forest — Huaye

This notebook is the authoritative Huaye Week 6 Isolation Forest analysis. It fits six independent models—one for each operating period—and uses only the six physical 15-minute rolling features. The earlier global 42-feature model and fixed top-percentage labels have been retired from the current implementation.

## Method

The dataset is split into `night_off`, `ramp_up`, `core_work`, `ramp_down`, `evening_off`, and `weekend`. A separate `IsolationForest` is fitted within each period using `contamination='auto'`. A row is labelled as a candidate anomaly only when that period model returns `predict(X) == -1`, equivalently `decision_function(X) < 0`. No fixed 3%, 5%, or 9% anomaly quota is imposed.

The output is an unsupervised candidate list, not confirmed faults. Baseline and LOF labels are retained only for descriptive comparison and are never model inputs.

In [1]:
from pathlib import Path
import sys
import pandas as pd


def locate_notebook_dir() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [
        cwd,
        cwd / 'ml',
        *cwd.parents,
        *[parent / 'ml' for parent in cwd.parents],
    ]
    for candidate in candidates:
        if (candidate / 'isolation_forest_period_auto_huaye.py').is_file():
            return candidate
    raise FileNotFoundError('Cannot locate isolation_forest_period_auto_huaye.py')


NOTEBOOK_DIR = locate_notebook_dir()
sys.path.insert(0, str(NOTEBOOK_DIR))

from isolation_forest_period_auto_huaye import (
    MODEL_FEATURES,
    MODEL_PARAMETERS,
    OUTPUT_DIR,
    OUTPUT_FILES,
    PERIOD_ORDER,
    run_pipeline,
)

print('Notebook dir:', NOTEBOOK_DIR)
print('Operating periods:', PERIOD_ORDER)
print('Physical features:', len(MODEL_FEATURES))
print('Model parameters:', MODEL_PARAMETERS)


Notebook dir: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\ml
Operating periods: ['night_off', 'ramp_up', 'core_work', 'ramp_down', 'evening_off', 'weekend']
Physical features: 6
Model parameters: {'n_estimators': 100, 'max_samples': 'auto', 'contamination': 'auto', 'max_features': 1.0, 'bootstrap': False, 'random_state': 42, 'n_jobs': -1}


## Fit Six Period Models and Save Outputs

The implementation preserves `timestamp + floor + zone`, aligns the unscaled physical rolling values and Gavin's LOF result by key, fits each period once, validates the automatic decision threshold, and writes full and anomaly-only outputs.

In [2]:
results, summaries, modelling_summary = run_pipeline()
print(f"Rows: {len(results):,}")
print(f"Calibrated candidate anomalies: {modelling_summary['anomaly_count']:,}")
print(f"Calibrated candidate anomaly rate: {modelling_summary['anomaly_rate_pct']:.4f}%")
print(f"Auto-threshold reference anomalies: {modelling_summary['auto_anomaly_count']:,}")
print(f"Auto-threshold reference anomaly rate: {modelling_summary['auto_anomaly_rate_pct']:.4f}%")
print(f"Persistent IF events: {modelling_summary['event_count']:,}")


Rows: 362,880
Calibrated candidate anomalies: 18,144
Calibrated candidate anomaly rate: 5.0000%
Auto-threshold reference anomalies: 54,868
Auto-threshold reference anomaly rate: 15.1201%
Persistent IF events: 134


## Results by Operating Period

The anomaly proportion is allowed to vary by period. These percentages are produced by each model's automatic decision boundary; they are not preselected quotas and are not estimates of confirmed fault prevalence.

In [3]:
period_summary = summaries['period_summary'].copy()
period_display_columns = [
    'operating_period', 'rows', 'anomaly_count', 'anomaly_rate_pct',
    'auto_anomaly_count', 'auto_anomaly_rate_pct', 'calibrated_threshold',
    'if_event_count', 'lof_overlap', 'baseline_overlap',
]
period_display_columns += [
    column for column in period_summary.columns
    if column.startswith('baseline_') or column.startswith('if_baseline_')
]
period_summary[[column for column in period_display_columns if column in period_summary.columns]]


,operating_period,rows,anomaly_count,anomaly_rate_pct,auto_anomaly_count,auto_anomaly_rate_pct,calibrated_threshold,if_event_count,lof_overlap,baseline_overlap,...,baseline_robust_z_score_label_count,if_baseline_robust_z_score_overlap_count,baseline_high_quantile_label_count,if_baseline_high_quantile_overlap_count,baseline_low_quantile_label_count,if_baseline_low_quantile_overlap_count,baseline_zero_aware_label_count,if_baseline_zero_aware_overlap_count,baseline_operational_label_count,if_baseline_operational_overlap_count
0,night_off,75600,3780,5.0,5980,7.910053,0.546409,50,439,269,...,5875,262,0,0,0,0,7,7,0,0
1,ramp_up,10800,540,5.0,2455,22.731481,0.567572,0,68,113,...,105,0,231,17,2008,96,0,0,0,0
2,core_work,97200,4860,5.0,17430,17.932099,0.569296,23,166,4417,...,6684,3837,4123,63,22371,4392,0,0,0,0
3,ramp_down,21600,1080,5.0,3010,13.935185,0.562671,12,81,586,...,1325,439,629,17,4550,462,32,0,0,0
4,evening_off,54000,2700,5.0,11559,21.405556,0.582992,25,414,284,...,2753,284,0,0,0,0,7,0,0,0
5,weekend,103680,5184,5.0,14434,13.921682,0.580720,24,85,568,...,3269,564,0,0,0,0,0,0,456,456


## Concrete Candidate Anomaly Values

The anomaly-only output contains the physical readings, unscaled 15-minute rolling values, scaled model inputs, anomaly score, decision function, and comparison labels. Higher `period_if_anomaly_score` means more unusual within the same operating period. Raw scores from separate period models should not be compared directly across periods.

In [4]:
baseline_detail_columns = [
    column for column in [
        'baseline_anomaly_count',
        'baseline_anomaly_methods',
        'baseline_anomaly_variables',
        'baseline_anomaly_directions',
    ]
    if column in results.columns
]
baseline_method_columns = [
    column for column in results.columns
    if column.startswith('baseline_') and column.endswith('_label') and column != 'baseline_anomaly_label'
]
example_columns = [
    'timestamp', 'floor', 'zone', 'operating_period',
    'total_AC_power_kw', 'light_power_kw', 'plug_power_kw',
    'temperature_c', 'humidity_pct', 'lux',
    'period_if_anomaly_score', 'period_if_threshold_score',
    'period_if_decision_function', 'period_if_auto_anomaly_label',
    'period_if_rank_within_period', 'lof_anomaly_label', 'baseline_anomaly_label',
] + baseline_detail_columns + baseline_method_columns
(
    results.loc[results['period_if_anomaly_label'].eq(1), example_columns]
    .sort_values(['operating_period', 'period_if_anomaly_score'], ascending=[True, False])
    .groupby('operating_period', observed=True)
    .head(3)
)


,timestamp,floor,zone,operating_period,total_AC_power_kw,light_power_kw,plug_power_kw,temperature_c,humidity_pct,lux,...,baseline_anomaly_label,baseline_anomaly_count,baseline_anomaly_methods,baseline_anomaly_variables,baseline_anomaly_directions,baseline_robust_z_score_label,baseline_high_quantile_label,baseline_low_quantile_label,baseline_zero_aware_label,baseline_operational_label
219364,2018-07-27 08:04:00,6,4,core_work,31.92,0.00,0.17,22.75,65.54,1.0,...,1,2,low_quantile_threshold,"light_power_kw,lux",low,0,0,1,0,0
219367,2018-07-27 08:07:00,6,4,core_work,31.84,0.00,0.04,22.66,65.23,1.0,...,1,2,low_quantile_threshold,"light_power_kw,lux",low,0,0,1,0,0
219365,2018-07-27 08:05:00,6,4,core_work,31.89,0.00,0.17,22.72,65.42,1.0,...,1,2,low_quantile_threshold,"light_power_kw,lux",low,0,0,1,0,0
133666,2018-07-30 19:46:00,6,2,evening_off,0.00,5.04,0.10,30.94,69.70,52.0,...,1,1,robust_z_score,temperature_c,high,1,0,0,0,0
133667,2018-07-30 19:47:00,6,2,evening_off,0.00,6.02,0.12,30.94,69.80,52.0,...,1,1,robust_z_score,temperature_c,high,1,0,0,0,0
133668,2018-07-30 19:48:00,6,2,evening_off,0.00,6.02,0.11,30.94,69.70,52.0,...,1,1,robust_z_score,temperature_c,high,1,0,0,0,0
236575,2018-08-08 06:55:00,6,4,night_off,1.44,3.40,0.11,26.53,67.40,23.0,...,0,0,none,none,none,0,0,0,0,0
236574,2018-08-08 06:54:00,6,4,night_off,1.42,10.44,0.26,26.55,67.13,23.0,...,0,0,none,none,none,0,0,0,0,0
236576,2018-08-08 06:56:00,6,4,night_off,1.44,3.40,0.17,26.51,66.94,23.0,...,0,0,none,none,none,0,0,0,0,0
248702,2018-08-16 17:02:00,6,4,ramp_down,2.37,3.91,0.96,26.21,57.74,40.0,...,0,0,none,none,none,0,0,0,0,0


## Descriptive Method Comparison

LOF still uses Gavin's existing fixed 0.05 contamination result, while the statistical baseline is a rule-based reference. Neither is ground truth, so overlap and Jaccard values do not measure accuracy.

In [5]:
display(summaries['comparison'])
display(summaries['event_summary'])
display(summaries['events'].head(20))


,comparison,if_anomaly_count,comparison_anomaly_count,overlap_count,union_count,jaccard_overlap,note
0,lof_k100_c050,18144,18144,1253,35035,0.035764,LOF uses its existing fixed contamination 0.05...
1,statistical_baseline_v3,18144,46659,6237,58566,0.106495,"Statistical baseline is a reference, not groun..."
2,statistical_baseline_v3_robust_z_score,18144,20011,5386,32769,0.164363,Method-specific statistical baseline reference...
3,statistical_baseline_v3_high_quantile,18144,4983,97,23030,0.004212,Method-specific statistical baseline reference...
4,statistical_baseline_v3_low_quantile,18144,28929,4950,42123,0.117513,Method-specific statistical baseline reference...
5,statistical_baseline_v3_zero_aware,18144,46,7,18183,0.000385,Method-specific statistical baseline reference...
6,statistical_baseline_v3_operational,18144,456,456,18144,0.025132,Method-specific statistical baseline reference...


,operating_period,point_anomaly_count,point_anomaly_rate_pct,event_count,persistent_event_points,event_point_capture_pct,mean_duration_minutes,max_duration_minutes,mean_anomaly_points,max_anomaly_score,baseline_overlap_points,lof_overlap_points
0,night_off,3780,5.0,50,1884,49.841270,37.740000,65.0,37.680000,0.767036,127,199
1,ramp_up,540,5.0,0,0,0.000000,0.000000,0.0,0.000000,0.000000,0,0
2,core_work,4860,5.0,23,3739,76.934156,162.869565,540.0,162.565217,0.669088,3707,110
3,ramp_down,1080,5.0,12,819,75.833333,68.250000,120.0,68.250000,0.706011,423,69
4,evening_off,2700,5.0,25,1746,64.666667,70.000000,138.0,69.840000,0.699522,244,263
5,weekend,5184,5.0,24,4866,93.865741,202.916667,615.0,202.750000,0.803150,529,71


,if_event_id,zone,operating_period,start_time,end_time,duration_minutes,anomaly_points,required_duration_minutes,max_gap_minutes,mean_anomaly_score,max_anomaly_score,min_decision_function,lof_overlap_points,baseline_overlap_points,baseline_overlap_pct
0,1,2,weekend,2018-08-04 09:36:00,2018-08-04 19:50:00,615,615,30,2,0.754219,0.803150,-0.303150,13,491,79.837398
1,2,4,night_off,2018-07-20 06:25:00,2018-07-20 06:59:00,35,35,30,2,0.696788,0.767036,-0.267036,0,0,0.000000
2,3,4,night_off,2018-07-26 06:30:00,2018-07-26 06:59:00,30,30,30,2,0.586978,0.757960,-0.257960,13,0,0.000000
3,4,2,night_off,2018-08-14 06:16:00,2018-08-14 06:59:00,44,44,30,2,0.665173,0.748072,-0.248072,29,14,31.818182
4,5,2,night_off,2018-08-24 06:19:00,2018-08-24 06:59:00,41,41,30,2,0.706791,0.745741,-0.245741,8,0,0.000000
5,6,2,night_off,2018-08-02 05:56:00,2018-08-02 06:59:00,64,64,30,2,0.721787,0.745412,-0.245412,6,0,0.000000
6,7,2,night_off,2018-07-26 06:28:00,2018-07-26 06:59:00,32,32,30,2,0.694883,0.735322,-0.235322,0,0,0.000000
7,8,2,night_off,2018-07-09 06:27:00,2018-07-09 06:59:00,33,33,30,2,0.691901,0.731752,-0.231752,4,0,0.000000
8,9,4,night_off,2018-07-19 06:29:00,2018-07-19 06:59:00,31,31,30,2,0.665137,0.717510,-0.217510,1,0,0.000000
9,10,2,weekend,2018-08-11 08:50:00,2018-08-11 17:32:00,523,523,30,2,0.669045,0.712013,-0.212013,11,0,0.000000


## Validation

The checks below confirm row/key preservation, six independent period models, exactly six physical inputs, complete scores, consistent `predict` and `decision_function` labels, non-fixed period rates, and exclusion of comparison labels from training.

In [6]:
validation = pd.DataFrame(modelling_summary['validation'])
assert validation['passed'].all()
validation

,check,passed,evidence
0,row_count_preserved,True,"rows=362,880"
1,expected_row_count,True,"rows=362,880"
2,unique_row_key,True,timestamp + floor + zone
3,six_period_models,True,"models=['core_work', 'evening_off', 'night_off..."
4,six_physical_features,True,features=['numerical__total_AC_power_kw_rollin...
5,scores_complete,True,missing=0
6,auto_decision_threshold_consistent,True,auto label=1 exactly when decision_function < 0
7,calibrated_period_rate,True,"{'night_off': 5.0, 'ramp_up': 5.0, 'core_work'..."
8,reference_labels_excluded,True,baseline and LOF labels are comparison-only
9,all_anomaly_rows_exportable,True,"anomalies=18,144"


## Generated Outputs

Only the new `if_period_auto_*` files are authoritative for Huaye's current Isolation Forest analysis. Presentation figures are intentionally outside the scope of this revision.

In [7]:
for name, filename in OUTPUT_FILES.items():
    print(f"{name}: {OUTPUT_DIR / filename}")

full_results: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_results_full.csv.gz
anomalies: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_anomalies.csv.gz
events: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_events.csv
period_summary: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_summary.csv
event_summary: D:\IRP\Artificial-Intelligence-for-Smart-Building-Log-Analysis\Artificial Intelligence for Smart-Building Log Analysis\outputs\isolation_forest_huaye\if_period_auto_event_summary.csv
zone_summary: D:\IRP\Artificial-I